# 2019韩国海关前20日原稿索引：四个月标题/附件身份修复

复用已备份的60份列表响应，仅抓取2019年1月两个精确官方页面；以附件名称辨别错误同题页面。


In [ ]:
import hashlib,json,os,pathlib,tarfile,tempfile,subprocess,shlex,urllib.request,urllib.parse,datetime,re
import pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.workflow import R
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2019_title_identity_repair_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
index=root/src['index_card'];manifest_path=root/src['index_request_manifest'];old=index/'qlib_recorder'/src['index_experiment_id']/src['index_recorder_id']/'artifacts/analysis/monthly_index'
for p,h in [(index/'summary.json',src['index_summary_sha256']),(manifest_path,src['index_request_manifest_sha256']),(old,src['index_monthly_artifact_sha256'])]:assert sha(p)==h
old_summary=json.loads((index/'summary.json').read_text());assert old_summary['source_archive']['sha256']==json.loads((index/'source_restore.json').read_text())['archive_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(index/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['index_experiment_id'],recorder_id=src['index_recorder_id']);assert rec.status=='FINISHED';monthly=rec.load_object('analysis/monthly_index').copy()
manifest=json.loads(manifest_path.read_text());assert len(monthly)==len(manifest['requests'])==60 and set(monthly.loc[monthly.accepted_count.ne(1),'month'])=={'2019-01','2019-05','2019-07','2019-08'}
repairs=[]
for month,identity in plan['fixed_saved_list_repairs'].items():
 receipt=next(x for x in manifest['requests'] if x['month']==month);items=[x for x in receipt['candidates'] if x['newsId']==identity];assert len(items)==1 and receipt['http_status']==200
 title=re.sub(r'\s+','',items[0]['title_excerpt']);month_number=int(month[-2:]);assert title.startswith(f'{month_number}.1') and '20' in title and '수출입현황' in title
 loc=monthly.month.eq(month);assert loc.sum()==1;monthly.loc[loc,'newsId']=identity;monthly.loc[loc,'source_url']=items[0]['href'];monthly.loc[loc,'accepted_count']=1;monthly.loc[loc,'parse_status']='repaired_saved_abbreviated_title';repairs.append({'month':month,'newsId':identity,'evidence':'exact_saved_official_list_response_and_fixed_abbreviated_title','title_excerpt':items[0]['title_excerpt']})
snapshot=root/'data/snapshots/kcs_early20_2019_title_identity_repair_20261023';snapshot.mkdir(parents=True,exist_ok=True);receipts=[];state={'status':'January_GET_started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'GET_completed':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
for url in plan['exact_new_GETs']:
 identity=urllib.parse.parse_qs(urllib.parse.urlsplit(url).query)['newsId'][0];start=datetime.datetime.now(datetime.timezone.utc).isoformat();state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');response=urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':plan['transport']['user_agent']}),timeout=plan['transport']['timeout_seconds_each']);body=response.read(plan['transport']['max_response_bytes_each']+1);status=response.status;headers=dict(response.headers.items());response.close();end=datetime.datetime.now(datetime.timezone.utc).isoformat();assert status==200 and len(body)<=plan['transport']['max_response_bytes_each'] and 'html' in headers.get('Content-Type','').lower();state['GET_completed']+=1
 (snapshot/(identity+'.response')).write_bytes(body);(snapshot/(identity+'.headers.json')).write_text(json.dumps(headers,ensure_ascii=False,indent=2)+'\n');soup=BeautifulSoup(body,'html.parser');attachments=[]
 for a in soup.find_all('a'):
  href=a.get('href') or '';name=a.get_text(' ',strip=True)
  if ('download.do' in href or 'fileId=' in href) and ('.hwp' in name.lower() or '.pdf' in name.lower()):attachments.append({'name':name,'href':urllib.parse.urljoin(url,href)})
 receipts.append({'newsId':identity,'url':url,'http_status':status,'started_utc':start,'ended_utc':end,'body_sha256':hashlib.sha256(body).hexdigest(),'body_bytes':len(body),'headers_sha256':sha(snapshot/(identity+'.headers.json')),'attachments':attachments})
assert state['GET_attempts']==state['GET_completed']==2
def valid(r):return [a for a in r['attachments'] if '190121' in re.sub(r'\s+','',a['name']) and '수출입현황' in re.sub(r'\s+','',a['name'])]
match=[r for r in receipts if len(valid(r))==1];assert len(match)==1 and all(len(valid(r))==0 for r in receipts if r not in match);chosen=match[0]['newsId'];rejected=next(r['newsId'] for r in receipts if r['newsId']!=chosen)
jan_receipt=next(x for x in manifest['requests'] if x['month']=='2019-01');jan_candidate=next(x for x in jan_receipt['accepted'] if x['newsId']==chosen);loc=monthly.month.eq('2019-01');assert loc.sum()==1;monthly.loc[loc,'newsId']=chosen;monthly.loc[loc,'source_url']=jan_candidate['href'];monthly.loc[loc,'accepted_count']=1;monthly.loc[loc,'parse_status']='repaired_original_attachment_identity';repairs.append({'month':'2019-01','chosen_newsId':chosen,'rejected_newsId':rejected,'valid_attachment':valid(match[0])[0]['name'],'rejected_attachment_names':[a['name'] for r in receipts if r['newsId']==rejected for a in r['attachments']]})
assert len(monthly)==60 and monthly.accepted_count.eq(1).all() and len(repairs)==4
original=manifest['requests'];old_map={x['month']:(x['accepted'][0]['newsId'] if x['accepted_count']==1 else None) for x in original}
for row in monthly.itertuples():
 if row.month not in {'2019-01','2019-05','2019-07','2019-08'}:assert row.newsId==old_map[row.month]
source_manifest={'plan_sha256':sha(card/'plan.json'),'January_page_receipts':receipts,'repairs':repairs};(snapshot/'request_manifest.json').write_text(json.dumps(source_manifest,ensure_ascii=False,indent=2)+'\n');files=sorted(p for p in snapshot.iterdir() if p.is_file());assert len(files)==5;settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-2019-repair-') as temp:
 archive=pathlib.Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest;source_archive={**receipt,'nas_path':remote,'members':5,'request_manifest_sha256':sha(snapshot/'request_manifest.json')}
with tempfile.TemporaryDirectory(prefix='kcs-2019-repair-cold-') as temp:
 copy=pathlib.Path(temp)/'source.tar.gz'
 with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True)
 assert copy.stat().st_size==source_archive['size_bytes'] and sha(copy)==digest
 with tarfile.open(copy,'r:gz') as t:
  for p in files:assert t.extractfile(str(p.relative_to(root))).read()==p.read_bytes()
source_restore={'status':'NAS_two_January_original_pages_headers_and_manifest_cold_read_exact','archive_sha256':digest,'members_checked':5};(card/'source_restore.json').write_text(json.dumps(source_restore,indent=2)+'\n')
result={'status':'official_portal_2018_2022_all_60_early20_page_leads_identified_source_only','plan_sha256':sha(card/'plan.json'),'months':60,'prior_unique':56,'repaired_months':repairs,'final_unique':60,'unchanged_original_rows':56,'GET_attempts':2,'source_archive':source_archive,'source_restore':'source_restore.json','SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,old_index_identity={'summary_sha256':src['index_summary_sha256'],'manifest_sha256':src['index_request_manifest_sha256']},source_archive=source_archive);rec.save_objects(artifact_path='analysis',repaired_index=monthly,January_pages=receipts,repairs=repairs,result=result);rec.log_metrics(unique_months=60,repaired_months=4,external_GETs=2,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'official_portal_2018_2022_early20_page_index_complete_original_attachments_numeric_PIT_pending','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':source_archive,'source_restore':'source_restore.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'verified':True},'external_GETs':2,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/repaired_index','analysis/January_pages','analysis/repairs','analysis/result']},'limitations':['Only government portal page identities are complete; 60 original attachments and semiconductor figures remain unqualified.','The current page bodies do not certify historical first-public versions or US decision-time availability.','No Stage1 factor, account, SOXX excess or forward strategy day.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'January_chosen':chosen,'rejected':rejected,'repaired':repairs,'source_sha256':digest,'recorder_id':rid,'Recorder_NAS_sha256':evidence['sha256']},ensure_ascii=False,indent=2))


## 中文结论

复用前版已存入NAS的60份官方海关列表响应，56个已唯一选中的月份逐行保持不变。2019-05/07/08 的列表标题分别简写为“5.1-20”“7.1~20日”“8.1-20”，现依旧响应中的唯一固定 ID `156332380`、`156342252`、`156346327` 机械补回，**没有再查这56个月**。

2019-01 的两个官方同题页面只按冻结的精确 URL 各GET一次。[`156313866`](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156313866) 所附文件是“190121 说明节关税行政支持对策”，与标题不符；[`156313813`](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156313813) 所附为 **“(190121)19년 1월(1-20일)_수출입현황.hwp”**，故只选择后者。两张原始页面、响应头、UTC接收时钟及SHA均保留在Yanxuan NAS，源归档 `519fcf76da276d9a3e091f131f2e7001d96478694922e07f91b87abeffab9c6b` 的五件对象和Recorder `b5c5e83a8dafccfb25add88ebf549fc8c3a97840dbd29fd6abec7f467fb8ba68` 的四件工件已限定冷恢复。

修复后的官方门户**页面身份索引覆盖2018–2022全部60个月**，Qlib Recorder `f5383ad0165a46cb80b2bba6b9ed3f89` 保存60行逐月索引和四次修复依据。决定 `official_portal_2018_2022_early20_page_index_complete_original_attachments_numeric_PIT_pending`。这是页面线索，不是60份附件的原版半导体金额：除已另存的2021/2022年10月样本外，原始附件、数值行、工作日调整、历史首次公开版本和美股决策时点仍需逐项资格审查。0 SOXX收益读取、0拟合、0资金账户、0合格策略前向日；稳定超过SOXX的目标仍未完成。
